In [1]:
import requests
import bs4
import pandas as pd
import time

In [2]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US, en;q=0.5'
}
asin = 'B09G9HD6PD' # ASIN for an iPhone 13 on Amazon.in

In [3]:
req = requests.get(f'https://www.amazon.in/product-reviews/{asin}/?pageNumber=1', headers=headers)

In [4]:
print(req)

<Response [200]>


In [5]:
soup = bs4.BeautifulSoup(req.text, 'html.parser')

In [6]:
soup.title

<title dir="ltr">Amazon Sign-In</title>

In [7]:
soup.select('div[data-hook="review"]')

[]

In [8]:
len(soup.select('div[data-hook="review"]'))

0

In [9]:
soup.select('div[data-hook="review"]')[0]

IndexError: list index out of range

In [10]:
soup.select('div[data-hook="review"]')[0].select_one('span.a-profile-name').text.strip()

IndexError: list index out of range

In [ ]:
soup.select('div[data-hook="review"]')[0].select_one('i[data-hook="review-star-rating"] span').text.strip()

In [ ]:
tag = soup.select('div[data-hook="review"]')[0].select_one('a[data-hook="review-title"] span:not([class])')
if not tag:
    tag = soup.select('div[data-hook="review"]')[0].select_one('span[data-hook="review-title"] span:not([class])')
tag.text.strip()

In [ ]:
soup.select('div[data-hook="review"]')[0].select_one('span[data-hook="review-body"] span').text.strip()

In [ ]:
customer_names = []
ratings = []
comment_tags = []
comments = []

for pg_num in range(1, 6): # Scraping first 5 pages
    req = requests.get(f'https://www.amazon.in/product-reviews/{asin}/?pageNumber={pg_num}', headers=headers)
    soup_pg = bs4.BeautifulSoup(req.text, 'html.parser')
    
    reviews = soup_pg.select('div[data-hook="review"]')
    
    for review in reviews:
        name = review.select_one('span.a-profile-name')
        customer_names.append(name.text.strip() if name else 'N/A')
        
        rating = review.select_one('i[data-hook="review-star-rating"] span')
        ratings.append(rating.text.strip() if rating else 'N/A')
        
        tag = review.select_one('a[data-hook="review-title"] span:not([class])')
        if not tag:
             tag = review.select_one('span[data-hook="review-title"] span:not([class])')
        comment_tags.append(tag.text.strip() if tag else 'N/A')
        
        comment = review.select_one('span[data-hook="review-body"] span')
        comments.append(comment.text.strip() if comment else 'N/A')
        
    time.sleep(2) # Delay to prevent getting blocked

In [ ]:
dic = {
    'Customer Name': customer_names,
    'Rating': ratings,
    'Comment Tag': comment_tags,
    'Review/Comment': comments
}

In [ ]:
df = pd.DataFrame(dic)

In [ ]:
df